## Implementation example

In [1]:
import torch

In [2]:
train_data = 'you need to know how to code'
word_set = set(train_data.split()) 
vocab = {word: i+2 for i, word in enumerate(word_set)} 
vocab['<unk>'] = 0 
vocab['<pad>'] = 1 
print(vocab)


{'know': 2, 'you': 3, 'need': 4, 'to': 5, 'code': 6, 'how': 7, '<unk>': 0, '<pad>': 1}


In [3]:
embedding_table = torch.FloatTensor([ [ 0.0, 0.0, 0.0], [ 0.0, 0.0, 0.0], 
[ 0.2, 0.9, 0.3], [ 0.1, 0.5, 0.7], 
[ 0.2, 0.1, 0.8], [ 0.4, 0.1, 0.1], 
[ 0.1, 0.8, 0.9], [ 0.6, 0.1, 0.1]])

In [4]:
sample = 'you need to run'.split()
idxes = [] 
for word in sample:
    try: idxes.append(vocab[word])
    except KeyError:
        idxes.append(vocab['<unk>'])

idxes = torch.LongTensor(idxes)
lookup_result = embedding_table[idxes, :] 
print(lookup_result)

tensor([[0.1000, 0.5000, 0.7000],
        [0.2000, 0.1000, 0.8000],
        [0.4000, 0.1000, 0.1000],
        [0.0000, 0.0000, 0.0000]])


## Sentiment Analysis (Movie review) in Pytorch

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
# from torchtext import data, datasets

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchtext\__init__.py:7: SyntaxWarning: invalid escape sequence '\ '
  "\n/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ \n"


OSError: [WinError 127] 지정된 프로시저를 찾을 수 없습니다

In [ ]:
BATCH_SIZE = 64
lr = 0.001
EPOCHS = 10
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

In [ ]:
TEXT = data.Field(sequential=True, batch_first=True, lower=True)
LABEL = data.Field(sequential=False, batch_first=True)
trainset, testset = datasets.IMDB.splits(TEXT, LABEL)
TEXT.build_vocab(trainset, min_freq=5)
LABEL.build_vocab(trainset)

In [ ]:
trainset, valset = trainset.split(split_ratio=0.8)
train_iter, val_iter, test_iter = data.BucketIterator.splits(
(trainset, valset, testset), batch_size=BATCH_SIZE,
shuffle=True, repeat=False)
vocab_size = len(TEXT.vocab)
n_classes = 2

In [ ]:
print("[학습셋]: %d [검증셋]: %d [테스트셋]: %d [단어수]: %d [클래스] %d" %(len(trainset),len(valset), len(testset), vocab_size, n_classes))

In [ ]:
class BasicGRU(nn.Module):
    def __init__(self, n_layers, hidden_dim, n_vocab, embed_dim, n_classes, dropout_p=0.2):
        super(BasicGRU, self).__init__()
        print("Building Basic GRU model...")
        self.n_layers = n_layers
        self.embed = nn.Embedding(n_vocab, embed_dim)
        self.hidden_dim = hidden_dim
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embed_dim, self.hidden_dim, num_layers=self.n_layers, batch_first=True)
        self.out = nn.Linear(self.hidden_dim, n_classes)

In [ ]:
def forward(self, x):
    x = self.embed(x)
    h_0 = self._init_state(batch_size=x.size(0))
    x, _ = self.gru(x, h_0) # [i, b, h]
    h_t = x[:,-1,:]
    self.dropout(h_t)
    logit = self.out(h_t) # [b, h] -> [b, o]
    return logit
def _init_state(self, batch_size=1):
    weight = next(self.parameters()).data
    return weight.new(self.n_layers, batch_size, self.hidden_dim).zero_()


In [ ]:
def train(model, optimizer, train_iter):
    model.train()
    for b, batch in enumerate(train_iter):
        x, y = batch.text.to(DEVICE), batch.label.to(DEVICE)
        y.data.sub_(1) # 레이블 값을 0과 1로 변환
        optimizer.zero_grad()
        logit = model(x)
        loss = F.cross_entropy(logit, y)
        loss.backward()
        optimizer.step()

In [ ]:
def evaluate(model, val_iter):
   model.eval()
    corrects, total_loss = 0, 0
    for batch in val_iter:
        x, y = batch.text.to(DEVICE), batch.label.to(DEVICE)
        y.data.sub_(1) # 레이블 값을 0과 1로 변환
        logit = model(x)
        loss = F.cross_entropy(logit, y, reduction='sum')
        total_loss += loss.item()
        corrects += (logit.max(1)[1].view(y.size()).data == y.data).sum()
    size = len(val_iter.dataset)
    avg_loss = total_loss / size
    avg_accuracy = 100.0 * corrects / size
    return avg_loss, avg_accuracy

In [ ]:
model = BasicGRU(1, 256, vocab_size, 128, n_classes, 0.5).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
best_val_loss = None
for e in range(1, EPOCHS+1):
    train(model, optimizer, train_iter)
    val_loss, val_accuracy = evaluate(model, val_iter)
    
    print("[이폭: %d] 검증 오차:%5.2f | 검증 정확도:%5.2f" % (e, val_loss, val_accuracy))
    
    if not best_val_loss or val_loss < best_val_loss:
        if not os.path.isdir("snapshot"):
            os.makedirs("snapshot")
        torch.save(model.state_dict(), './snapshot/txtclassification.pt')
        best_val_loss = val_loss